# Notebook 02 - Pré-processamento de Texto

**Entrada:** `data/processed/zomato_reviews_clean.csv` (3.370 reviews)  
**Saída:** `data/processed/zomato_reviews_processed.csv` (+ coluna `review_processed`)

**Pipeline:**  
`review` → `clean_text` → `tokenize` → `remove_stopwords` → `lemmatize` → `review_processed`

In [2]:
import sys
sys.path.insert(0, '..')

import time
import pandas as pd
import nltk
from collections import Counter
from itertools import chain

for resource in ['stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

from src.preprocessing import (
    preprocess_pipeline, clean_text, tokenize, remove_stopwords, lemmatize
)

print('Setup concluido.')

Setup concluido.


## 1. Carregar dataset

In [3]:
df = pd.read_csv('../data/processed/zomato_reviews_clean.csv')
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
print()
print(df['sentiment'].value_counts())
df.head(3)

Shape: (3370, 3)
Colunas: ['rating', 'review', 'sentiment']

sentiment
positivo    1642
negativo    1439
neutro       289
Name: count, dtype: int64


,rating,review,sentiment
0,5,"best biryani , so supportive staff of outlet ,...",positivo
1,4,delivery boy was very decent and supportive.👌👍,positivo
2,1,"worst biryani i have tasted in my life, half o...",negativo


## 2. Pipeline de pré-processamento

**preprocessing.py** 

**Stopwords:** lista NLTK English - corpus predominantemente em inglês (decisão de idioma documentada no nb01). Tokens Hinglish transliterados passam pelo filtro por não constarem na lista inglesa.

**Limitação conhecida - lemmatização sem POS tags:** `WordNetLemmatizer.lemmatize(t)` usa `pos='n'` (substantivo) por padrão. Verbos não são reduzidos à forma base: `ordered` permanece `ordered` em vez de `order`. Resultado: vocabulário ligeiramente inflado, dois tokens para o mesmo lexema, aceitável para baseline TF-IDF.

In [4]:
example_idx = 1  # review com emoji para testar limpeza
original = df['review'].iloc[example_idx]

step1 = clean_text(original)
step2 = tokenize(step1)
step3 = remove_stopwords(step2)
step4 = lemmatize(step3)

print(f'[0] Original:       {original}')
print(f'[1] clean_text:     {step1}')
print(f'[2] tokenize:       {step2}')
print(f'[3] sem stopwords:  {step3}')
print(f'[4] lemmatizado:    {step4}')
print(f'\nResumo: {len(step2)} tokens -> {len(step4)} apos filtragem')

[0] Original:       delivery boy was very decent and supportive.👌👍
[1] clean_text:     delivery boy was very decent and supportive
[2] tokenize:       ['delivery', 'boy', 'was', 'very', 'decent', 'and', 'supportive']
[3] sem stopwords:  ['delivery', 'boy', 'decent', 'supportive']
[4] lemmatizado:    ['delivery', 'boy', 'decent', 'supportive']

Resumo: 7 tokens -> 4 apos filtragem


## 3. Aplicar pipeline ao dataset

In [5]:
start = time.time()
df['tokens'] = df['review'].apply(preprocess_pipeline)
df['review_processed'] = df['tokens'].apply(' '.join)
elapsed = time.time() - start

print(f'Pipeline aplicada em {elapsed:.1f}s | Shape: {df.shape}')
df[['review', 'review_processed']].head(5)

Pipeline aplicada em 0.1s | Shape: (3370, 5)


,review,review_processed
0,"best biryani , so supportive staff of outlet ,...",best biryani supportive staff outlet personali...
1,delivery boy was very decent and supportive.👌👍,delivery boy decent supportive
2,"worst biryani i have tasted in my life, half o...",worst biryani tasted life half biryani dustbin
3,all food is good and tasty . will order again ...,food good tasty order lot explore bawarchi menu
4,shandar zabardast zindabad .. good going bawar...,shandar zabardast zindabad good going bawarchi...


## 4. Diagnóstico pós-processamento

In [6]:
df['n_words_orig'] = df['review'].str.split().str.len()
df['n_tokens_proc'] = df['tokens'].str.len()

stats = (
    df.groupby('sentiment')[['n_words_orig', 'n_tokens_proc']]
    .agg(['mean', 'median'])
    .round(1)
)
print('Tokens por classe — antes x depois do pre-processamento:')
print(stats)

Tokens por classe — antes x depois do pre-processamento:
          n_words_orig        n_tokens_proc       
                  mean median          mean median
sentiment                                         
negativo          12.5    9.0           7.5    5.0
neutro            12.3    9.0           7.5    5.0
positivo          13.4   10.0           8.0    6.0


In [7]:
all_tokens = list(chain.from_iterable(df['tokens']))
vocab = set(all_tokens)

orig_mean = df['n_words_orig'].mean()
proc_mean = df['n_tokens_proc'].mean()
reduction_pct = (1 - proc_mean / orig_mean) * 100

print(f'Total de tokens (com repeticao): {len(all_tokens):,}')
print(f'Vocabulario unico:               {len(vocab):,}')
print(f'Reducao media por review:        {orig_mean:.1f} -> {proc_mean:.1f} tokens ({reduction_pct:.1f}% menos)')

Total de tokens (com repeticao): 26,057
Vocabulario unico:               4,189
Reducao media por review:        12.9 -> 7.7 tokens (40.0% menos)


In [8]:
for label in ['positivo', 'neutro', 'negativo']:
    subset = list(chain.from_iterable(df[df['sentiment'] == label]['tokens']))
    top20 = Counter(subset).most_common(20)
    tokens_str = '  '.join([w + '(' + str(c) + ')' for w, c in top20])
    print(f'\n{label.upper()}:')
    print(tokens_str)


POSITIVO:
food(367)  good(345)  taste(311)  order(216)  bad(153)  quality(134)  delivery(120)  quantity(111)  time(102)  service(100)  like(96)  ordered(95)  restaurant(93)  le(88)  also(77)  worst(76)  test(73)  money(70)  best(68)  experience(66)

NEUTRO:
food(60)  taste(57)  good(54)  bad(45)  order(33)  delivery(29)  money(28)  restaurant(24)  quality(22)  best(22)  service(19)  also(17)  worst(16)  quantity(15)  time(14)  nice(13)  please(13)  tha(13)  le(13)  late(12)

NEGATIVO:
food(277)  good(270)  taste(237)  order(197)  bad(135)  quality(112)  delivery(105)  like(99)  quantity(96)  test(94)  restaurant(92)  service(88)  time(82)  ordered(79)  worst(76)  money(76)  chicken(68)  best(66)  le(63)  also(63)


**Alerta de negação - `bad` e `good` invertidos entre classes:**  
`bad` aparece 153× no positivo e `good` 270× no negativo - padrão que unigramas TF-IDF não conseguem resolver. Contextos típicos: *"not bad"*, *"not that bad"* (positivo) e *"not good enough"*, *"never good"* (negativo). Evidência direta para `ngram_range=(1,2)` obrigatório no nb03.

**Achado - `test` não é artefato de preprocessing:**  
`test` aparece 94× no negativo e 73× no positivo. Hipótese: escrita fonética Hinglish para *taste*, prática comum em avaliações de delivery no subcontinente indiano (ex: *"test is bad"* = *"taste is bad"*). Não há conflito semântico com o inglês "test" neste contexto de restaurante. Token mantido, o TF-IDF tratará como feature normal.

### Investigação de tokens suspeitos

`le` aparece 88× no positivo e 63× no negativo — volume alto para um token de 2 chars.

In [9]:
from nltk.corpus import stopwords as nltk_sw
from nltk.stem import WordNetLemmatizer

lem = WordNetLemmatizer()
sw_en = set(nltk_sw.words('english'))

print("--- 'less' é stopword NLTK? ---")
print(f"  'less' in stopwords: {'less' in sw_en}  <- passa pelo filtro")
print()
print("--- Comportamento do lemmatizer com 'less' ---")
print(f"  lemmatize('less', pos='n') [padrão]: {lem.lemmatize('less', pos='n')!r}  <- artefato")
print(f"  lemmatize('less', pos='r') [adv]:    {lem.lemmatize('less', pos='r')!r}  <- correto com POS")
print()
print("--- Pipeline passo a passo — review real com LESS ---")
review_less = 'HAD GIVEN VERY VERY VERY VERY LESS CHILLI FLAKES AND NO OREGANO AT ALL'
s1 = clean_text(review_less)
s2 = tokenize(s1)
s3 = remove_stopwords(s2)
s4 = lemmatize(s3)
print(f'  original:          {review_less}')
print(f'  clean_text:        {s1}')
print(f'  tokenize:          {s2}')
print(f'  remove_stopwords:  {s3}  <- "less" NAO removido')
print(f'  lemmatize(pos=n):  {s4}  <- "less" vira "le"')


--- 'less' é stopword NLTK? ---
  'less' in stopwords: False  <- passa pelo filtro

--- Comportamento do lemmatizer com 'less' ---
  lemmatize('less', pos='n') [padrão]: 'le'  <- artefato
  lemmatize('less', pos='r') [adv]:    'less'  <- correto com POS

--- Pipeline passo a passo — review real com LESS ---
  original:          HAD GIVEN VERY VERY VERY VERY LESS CHILLI FLAKES AND NO OREGANO AT ALL
  clean_text:        had given very very very very less chilli flakes and no oregano at all
  tokenize:          ['had', 'given', 'very', 'very', 'very', 'very', 'less', 'chilli', 'flakes', 'and', 'no', 'oregano', 'at', 'all']
  remove_stopwords:  ['given', 'less', 'chilli', 'flakes', 'oregano']  <- "less" NAO removido
  lemmatize(pos=n):  ['given', 'le', 'chilli', 'flake', 'oregano']  <- "less" vira "le"


**Conclusão pós-investigação:** é um artefato da lemmatização sem POS. `WordNetLemmatizer().lemmatize('less', pos='n')` retorna `'le'` - o lemmatizer interpreta "less" como substantivo plural e produz uma forma inventada. Verificado: `'less' not in nltk.corpus.stopwords.words('english')` - `less` passa pelo filtro de stopwords e chega ao lemmatizer. Todas as amostras confirmam: reviews com `le` contêm a palavra "less" (ex: *"taste less"*, *"quantity are less"*).

O sinal semântico de `less` está preservado sob o token `le` - o modelo vai aprender a correlação, mas a interpretabilidade fica comprometida. Na análise de feature importance do nb03, `le` = "less".

Tokens curtos adicionais (`nd`, `hi`, `ka`, `ki`, `ke`, `ho`) são partículas Hinglish.

In [10]:
# Tokens de 2 chars mais frequentes - candidatos a ruído
short_tokens = [(t, c) for t, c in Counter(all_tokens).most_common() if len(t) == 2]
print('Top tokens de 2 chars:')
for tok, cnt in short_tokens[:10]:
    print(f'  {repr(tok)}: {cnt}x')

# Amostras de reviews com token 'le'
print('\nAmostras de reviews com token "le":')
le_mask = df['tokens'].apply(lambda x: 'le' in x)
print(df[le_mask][['review', 'sentiment']].head(8).to_string())

Top tokens de 2 chars:
  'le': 164x
  'hi': 36x
  'nd': 34x
  'se': 34x
  'ka': 31x
  'ok': 29x
  'ki': 24x
  'go': 23x
  'ke': 22x
  'ho': 18x

Amostras de reviews com token "le":
                                                                                                                       review sentiment
19                                                                             Taste less nd took 55 min to deliver the order  negativo
41                                                                              quality not much good and quantity  are less   negativo
50                                   HAD GIVEN VERY VERY VERY VERY LESS CHILLI FLAKES AND NO OREGANO AT ALL VERY DISSAPOINTED  positivo
77                  very bad taste taste less saltless ,they don't even accept food instructions, no cutlery and very costly   positivo
125                                                                                           size of both item is very less   negativo
144

In [11]:
# Reviews que ficaram sem tokens (Hindi/Gujarati puro ou frases totalmente de stopwords)
empty_mask = df['n_tokens_proc'] == 0
print(f'Reviews sem tokens apos preprocessing: {empty_mask.sum()}')
print()
print(df[empty_mask][['review', 'sentiment']].to_string())

Reviews sem tokens apos preprocessing: 14

                                                                                                                                                                                                              review sentiment
129                                                                                                                                                                                                  जादा ठिक नही था  positivo
130                                                                                                                                                                                                     अच्छा नही था  positivo
135                                                                                                                                            🙏🏻જય સ્વામિનારાયણ<br/><br/>બહુ સરસ કોલ્ડ કોકો ક્વોલિટી બહુ સારી આભાર     neutro
457                                                              

## 5. Salvar dataset processado

Removidas as reviews sem tokens - sem sinal semântico para TF-IDF.

In [12]:
import csv

cols_to_save = ['rating', 'review', 'sentiment', 'review_processed']
output_path = '../data/processed/zomato_reviews_processed.csv'

# Strings que pandas interpreta como NaN ao ler CSV
_PANDAS_NA = {'nan', 'NaN', 'NA', 'N/A', 'n/a', 'null', 'NULL', '', '#N/A', '#NA'}

n_before = len(df)
n_empty_tokens = (df['n_tokens_proc'] == 0).sum()
n_nan_string = df[df['n_tokens_proc'] > 0]['review_processed'].isin(_PANDAS_NA).sum()
print(f'  tokens vazios:       {n_empty_tokens}')
print(f'  string nan residual: {n_nan_string}')

df_final = df[
    (df['n_tokens_proc'] > 0) &
    (~df['review_processed'].isin(_PANDAS_NA))
].copy()
n_dropped = n_before - len(df_final)
print(f'Total removidas: {n_dropped} reviews ({n_dropped/n_before*100:.1f}%)')
print(f'Shape salvo: {df_final[cols_to_save].shape}')
print(df_final['sentiment'].value_counts())

# QUOTE_NONNUMERIC evita que strings como 'nan' sejam lidas de volta como NaN
df_final[cols_to_save].to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC)

print(f'\nSalvo: {output_path}')

  tokens vazios:       14
  string nan residual: 1
Total removidas: 15 reviews (0.4%)
Shape salvo: (3355, 4)
sentiment
positivo    1636
negativo    1434
neutro       285
Name: count, dtype: int64

Salvo: ../data/processed/zomato_reviews_processed.csv
